# Entrenamiento del Generador de Poemas 🎭

## Instrucciones
1. Asegúrate de tener GPU activa: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4**
2. Sube el archivo `poems.csv` cuando la celda te lo pida
3. Ejecuta todas las celdas en orden
4. Al final descarga `inner_model.keras` y `meta.pkl` y ponlos en la carpeta `models/` de tu proyecto

In [ ]:
# Celda 1 — Verificar GPU
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('GPU disponible:', gpus if gpus else '❌ No hay GPU — revisa el tipo de entorno de ejecución')

In [ ]:
# Celda 2 — Subir poems.csv
from google.colab import files
uploaded = files.upload()   # selecciona dataset/poems.csv de tu computador
print('Archivo recibido:', list(uploaded.keys()))

In [ ]:
# Celda 3 — Código del generador (copia exacta de modules/generator.py)
import os, re, pickle
import numpy as np
import pandas as pd
import tensorflow as tf

SEQ_LENGTH    = 100
EMBEDDING_DIM = 16
GRU_UNITS     = 128
BATCH_SIZE    = 32
EPOCHS        = 10
MODEL_DIR     = '/content/models'
CSV_PATH      = 'poems.csv'


def _clean(text: str) -> str:
    text = text.lower()
    text = re.sub(r'[^a-zñáéíóúüç\s]', '', text)
    return re.sub(r'\s+', ' ', text).strip()


def _to_dataset(sequence, length, shuffle=False, seed=None, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices(sequence)
    ds = ds.window(length + 1, shift=1, drop_remainder=True)
    ds = ds.flat_map(lambda w: w.batch(length + 1))
    if shuffle:
        ds = ds.shuffle(100_000, seed=seed)
    ds = ds.batch(batch_size)
    return ds.map(lambda w: (w[:, :-1], w[:, 1:])).prefetch(1)


class PoetryGenerator:
    def __init__(self):
        self.text_vec_layer = None
        self._inner_model   = None
        self._poem_model    = None
        self.n_tokens       = 0
        self.authors_text   = {}

    def _build_inner_model(self):
        tf.random.set_seed(42)
        model = tf.keras.Sequential([
            tf.keras.layers.Embedding(input_dim=self.n_tokens, output_dim=EMBEDDING_DIM),
            tf.keras.layers.GRU(GRU_UNITS, return_sequences=True),
            tf.keras.layers.Dense(self.n_tokens, activation='softmax'),
        ])
        model.compile(loss='sparse_categorical_crossentropy', optimizer='nadam', metrics=['accuracy'])
        return model

    def _build_poem_model(self):
        return tf.keras.Sequential([
            self.text_vec_layer,
            tf.keras.layers.Lambda(lambda X: X - 2),
            self._inner_model,
        ])

    def train(self, csv_path=CSV_PATH, epochs=EPOCHS):
        df = pd.read_csv(csv_path)

        for author, group in df.groupby('author'):
            self.authors_text[author] = _clean(
                ' '.join(group['content'].fillna('').astype(str).tolist())
            )

        full_text = _clean(' '.join(df['content'].fillna('').astype(str).tolist()))

        self.text_vec_layer = tf.keras.layers.TextVectorization(split='character', standardize='lower')
        self.text_vec_layer.adapt([full_text])

        encoded = self.text_vec_layer([full_text])[0]
        encoded -= 2
        self.n_tokens = self.text_vec_layer.vocabulary_size() - 2

        print(f'Vocabulary size: {self.n_tokens} characters')
        print(f'Dataset size   : {len(encoded)} encoded chars')

        split     = int(len(encoded) * 0.9)
        train_set = _to_dataset(encoded[:split], SEQ_LENGTH, shuffle=True, seed=42)
        valid_set = _to_dataset(encoded[split:],  SEQ_LENGTH)

        self._inner_model = self._build_inner_model()

        os.makedirs(MODEL_DIR, exist_ok=True)
        ckpt = tf.keras.callbacks.ModelCheckpoint(
            os.path.join(MODEL_DIR, 'best_model.keras'),
            monitor='val_accuracy', save_best_only=True,
        )
        self._inner_model.fit(train_set, validation_data=valid_set, epochs=epochs, callbacks=[ckpt])
        self._poem_model = self._build_poem_model()

    def save(self, model_dir=MODEL_DIR):
        os.makedirs(model_dir, exist_ok=True)
        self._inner_model.save(os.path.join(model_dir, 'inner_model.keras'))
        with open(os.path.join(model_dir, 'meta.pkl'), 'wb') as f:
            pickle.dump({
                'vocab':        self.text_vec_layer.get_vocabulary(),
                'n_tokens':     self.n_tokens,
                'authors_text': self.authors_text,
            }, f)

print('✅ Código cargado correctamente')

In [ ]:
# Celda 4 — Entrenar (con GPU T4 tarda ~20-30 min)
gen = PoetryGenerator()
gen.train()
gen.save()
print('\n✅ Modelo entrenado y guardado en', MODEL_DIR)

In [ ]:
# Celda 5 — Prueba rápida de generación
import numpy as np

def next_char(model, text_vec_layer, text, temperature=1.0):
    y_proba = model.predict([text], verbose=0)[0, -1:]
    rescaled = tf.math.log(y_proba) / temperature
    char_id  = tf.random.categorical(rescaled, num_samples=1)[0, 0]
    return text_vec_layer.get_vocabulary()[char_id + 2]

def extend_text(model, text_vec_layer, authors_text, author, n_chars=300, temperature=0.8):
    corpus = authors_text.get(author, '')
    start  = np.random.randint(0, max(1, len(corpus) - 100))
    window = corpus[start:start + 100]
    result = []
    for _ in range(n_chars):
        ch = next_char(model, text_vec_layer, window[-100:], temperature)
        result.append(ch)
        window += ch
    return ''.join(result)

# Prueba con Pablo Neruda
author = 'Pablo Neruda'
poem   = extend_text(gen._poem_model, gen.text_vec_layer, gen.authors_text, author)
print(f'--- Al estilo de {author} ---')
print(poem)

In [ ]:
# Celda 6 — Descargar los archivos del modelo
from google.colab import files

files.download(f'{MODEL_DIR}/inner_model.keras')
files.download(f'{MODEL_DIR}/meta.pkl')

print('\n📥 Descarga iniciada.')
print('Coloca ambos archivos en la carpeta  models/  de tu proyecto local.')